# 03 - Multi-Trial Benchmark Evaluations (Champions & Classical Baselines)

This notebook consolidates the **Post-Evolution Benchmark Evaluation Pipeline** into a unified, reproducible workflow:
1. **Champion Extraction & Balance Check**: Selects top-performing candidate algorithms per problem condition from SQLite (`data/db.sqlite3`) and exports `data/champions.json`.
2. **Pre-Flight Diagnostic Audit**: Scans existing IOH traces in `results/ioh_traces/` and reports workload status.
3. **LLaMEA Champions Benchmark Execution**: Executes $N=10$ independent evaluations per champion with IOHprofiler instrumentation.
4. **Classical Baselines Benchmark Execution**: Executes $N=10$ independent evaluations for baseline optimizers (`CMA-ES`, `DE`, `PSO`).

In [29]:
# Setup paths and services
# Ensure project root src/ is in sys.path
import os
import sys
from pathlib import Path

cwd = Path('.').resolve()
root_dir = cwd.parent if cwd.name == 'notebooks' else cwd
src_dir = root_dir / 'src'
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

%load_ext autoreload
%autoreload 2

import json
import pandas as pd
from IPython.display import display, HTML

from shared.config import DATA_DIR, RESULTS_DIR
from shared.database import create_db_session_factory
from benchmarking.infra.io.trace_repository import EvaluationStateRepository, IOHTraceReader
from benchmarking.infra.storage import (
    EvaluationConfigRepository,
    ChampionsReadRepository,
    SQLiteSynthesisReadRepository,
)
from benchmarking.application.selection_service import ChampionSelectionService
from benchmarking.application.evaluation_service import EvaluationService
from benchmarking.domain.services.baselines import BASELINES
from benchmarking.infra.logging import EvaluationLogger

session_factory = create_db_session_factory()
sqlite_repo = SQLiteSynthesisReadRepository(session_factory)
champions_repo = ChampionsReadRepository(session_factory)

champ_service = ChampionSelectionService(sqlite_repo=sqlite_repo, champions_repo=champions_repo)
trace_repo = IOHTraceReader()
state_repo = EvaluationStateRepository()
config_repo = EvaluationConfigRepository()
logger = EvaluationLogger()
eval_service = EvaluationService(
    sqlite_repo=sqlite_repo,
    champions_repo=champions_repo,
    trace_repo=trace_repo,
    state_repo=state_repo,
    config_repo=config_repo,
    logger=logger,
)
CHAMPIONS_PATH = DATA_DIR / 'champions.json'

# Evaluation settings & matrix conditions loaded directly from configs/benchmark.toml

print('✅ Benchmark evaluation environment initialized.')

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
✅ Benchmark evaluation environment initialized.


## 1. Champion Selection & Experiment Balance Check
Query the synthesis database, inspect run balance, and export problem-specific champions to `data/champions.json`.

In [30]:
summary, total_completed = champ_service.get_experiment_balance()

if summary.empty:
    raise RuntimeError('No completed experiments found in database. Please run Notebook 02 first.')

print(f'== Total Completed Experiments: {total_completed} ==')
print(summary.to_string(index=False))

# Export champions
champions_dict, df_champions = champ_service.export_champions(CHAMPIONS_PATH)
print(f'\n✅ Exported {len(df_champions)} champions to {CHAMPIONS_PATH}')


== Total Completed Experiments: 292 ==
 problem_id  dim  noise_std                 mode prompt_strategy                               llm_name  completed_count
          1    2       0.00 ExperimentMode.CLEAN        baseline qwen2.5-coder-14b-instruct-q4_k_m.gguf                2
          1    2       0.00 ExperimentMode.CLEAN        baseline  qwen2.5-coder-7b-instruct-q4_k_m.gguf                1
          1    2       0.00 ExperimentMode.CLEAN          guided qwen2.5-coder-14b-instruct-q4_k_m.gguf                1
          1    2       0.00 ExperimentMode.CLEAN          guided  qwen2.5-coder-7b-instruct-q4_k_m.gguf                1
          1    2       0.00 ExperimentMode.CLEAN        thinking qwen2.5-coder-14b-instruct-q4_k_m.gguf                1
          1    2       0.00 ExperimentMode.CLEAN        thinking  qwen2.5-coder-7b-instruct-q4_k_m.gguf                1
          1    2       0.00 ExperimentMode.CLEAN   vectorization qwen2.5-coder-14b-instruct-q4_k_m.gguf           

## 2. Pre-Flight Diagnostic Audit (Champions & Baselines Workload)
Audit pending vs. completed evaluation runs across all experimental conditions.

In [31]:
# 1. Load champions
with open(CHAMPIONS_PATH, "r", encoding="utf-8") as f:
    champions_raw = json.load(f)
champions_flat = eval_service.champions_repo.get_champions_flat(champions_raw)

# ── Embedded HTML Dashboard Helper (Notebook-Local UI) ─────────────────────────
def render_html_dashboard(
    df_audit: pd.DataFrame,
    title: str = "Benchmark Evaluation Pre-Flight Audit",
    subtitle: str = "Real-time status of empirical evaluation runs",
    group_column: str = "model",
) -> str:
    """Render responsive HTML pre-flight dashboard for Jupyter display."""
    if df_audit.empty:
        return "<div>No audit data available.</div>"

    grp_col = (
        group_column
        if group_column in df_audit.columns
        else ("solver" if "solver" in df_audit.columns else df_audit.columns[0])
    )

    from benchmarking.domain.services.resolvers import get_clean_model_label

    summary_rows = []
    for name, grp in df_audit.groupby(grp_col):
        clean_name = get_clean_model_label(str(name)) if grp_col == "model" else str(name).upper()
        total = len(grp)
        completed = len(grp[grp["status"] == "COMPLETED"])
        pending = len(grp[grp["status"] == "PENDING"])
        needs_rerun = len(grp[grp["status"] == "NEEDS_RERUN"])
        missing_code = len(grp[grp["status"] == "MISSING_CODE"])
        to_run_mask = grp["status"].isin(["PENDING", "NEEDS_RERUN"])
        if "is_filtered" in grp.columns:
            to_run_mask = to_run_mask & (~grp["is_filtered"])
        to_run = len(grp[to_run_mask])
        pct = (completed / total * 100) if total > 0 else 0.0
        summary_rows.append({
            "Group": clean_name,
            "Total Tasks": total,
            "Completed": completed,
            "Pending": pending,
            "Needs Rerun": needs_rerun,
            "Missing Code": missing_code,
            "Queue to Run": to_run,
            "Progress (%)": pct,
        })

    df_summary = pd.DataFrame(summary_rows)
    total_queue = int(df_summary["Queue to Run"].sum())
    total_completed = int(df_summary["Completed"].sum())
    total_tasks = int(df_summary["Total Tasks"].sum())
    overall_pct = (total_completed / total_tasks * 100) if total_tasks > 0 else 0.0

    html = f"""
<div style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; max-width: 950px; margin: 15px 0;">
    <div style="display: flex; align-items: center; justify-content: space-between; margin-bottom: 16px; border-bottom: 2px solid #E2E8F0; padding-bottom: 8px;">
        <div>
            <h2 style="margin: 0; color: #0F172A; font-size: 20px; font-weight: 700; display: flex; align-items: center; gap: 8px;">
                🚀 {title}
            </h2>
            <p style="margin: 4px 0 0 0; color: #64748B; font-size: 13px;">{subtitle}</p>
        </div>
        <div style="background: #EEF2F6; padding: 6px 12px; border-radius: 20px; font-size: 12px; font-weight: 600; color: #334155;">
            {total_tasks} Target Conditions
        </div>
    </div>

    <div style="display: grid; grid-template-columns: repeat(4, 1fr); gap: 12px; margin-bottom: 20px;">
        <div style="background: linear-gradient(135deg, #1E293B 0%, #0F172A 100%); padding: 14px 18px; border-radius: 10px; color: white; box-shadow: 0 4px 6px -1px rgba(0,0,0,0.1);">
            <div style="font-size: 11px; text-transform: uppercase; letter-spacing: 0.05em; color: #94A3B8;">Total Targets</div>
            <div style="font-size: 24px; font-weight: 700; color: #F8FAFC; margin-top: 4px;">{total_tasks}</div>
            <div style="font-size: 11px; color: #64748B; margin-top: 2px;">Across {len(df_summary)} Categories</div>
        </div>
        <div style="background: linear-gradient(135deg, #065F46 0%, #047857 100%); padding: 14px 18px; border-radius: 10px; color: white; box-shadow: 0 4px 6px -1px rgba(0,0,0,0.1);">
            <div style="font-size: 11px; text-transform: uppercase; letter-spacing: 0.05em; color: #A7F3D0;">Completed & Valid</div>
            <div style="font-size: 24px; font-weight: 700; color: #ECFDF5; margin-top: 4px;">{total_completed}</div>
            <div style="font-size: 11px; color: #D1FAE5; margin-top: 2px;">{overall_pct:.1f}% Overall Progress</div>
        </div>
        <div style="background: linear-gradient(135deg, #C2410C 0%, #9A3412 100%); padding: 14px 18px; border-radius: 10px; color: white; box-shadow: 0 4px 6px -1px rgba(0,0,0,0.1);">
            <div style="font-size: 11px; text-transform: uppercase; letter-spacing: 0.05em; color: #FED7AA;">Queue to Run</div>
            <div style="font-size: 24px; font-weight: 700; color: #FFF7ED; margin-top: 4px;">{total_queue}</div>
            <div style="font-size: 11px; color: #FFEDD5; margin-top: 2px;">Pending / Needs Rerun</div>
        </div>
        <div style="background: linear-gradient(135deg, #4338CA 0%, #3730A3 100%); padding: 14px 18px; border-radius: 10px; color: white; box-shadow: 0 4px 6px -1px rgba(0,0,0,0.1);">
            <div style="font-size: 11px; text-transform: uppercase; letter-spacing: 0.05em; color: #C7D2FE;">Overall Status</div>
            <div style="font-size: 24px; font-weight: 700; color: #EEF2FF; margin-top: 4px;">{overall_pct:.0f}%</div>
            <div style="font-size: 11px; color: #E0E7FF; margin-top: 2px;">Completion Rate</div>
        </div>
    </div>

    <div style="background: #F8FAFC; border: 1px solid #E2E8F0; border-radius: 10px; padding: 16px; margin-bottom: 8px;">
        <div style="font-size: 13px; font-weight: 700; color: #1E293B; margin-bottom: 12px; text-transform: uppercase; letter-spacing: 0.04em;">
            Completion Progress Breakdown
        </div>
"""
    for _, row in df_summary.iterrows():
        grp_name = row["Group"]
        tot = int(row["Total Tasks"])
        comp = int(row["Completed"])
        pend = int(row["Pending"])
        rerun = int(row["Needs Rerun"])
        q = int(row["Queue to Run"])
        pct = float(row["Progress (%)"])
        color = "#10B981" if pct > 75 else ("#3B82F6" if pct > 25 else "#F59E0B")
        rerun_pct = (rerun / tot * 100) if tot > 0 else 0

        html += f"""
        <div style="margin-bottom: 14px;">
            <div style="display: flex; justify-content: space-between; font-size: 13px; font-weight: 600; color: #334155; margin-bottom: 4px;">
                <span>⚙️ <strong style="color: #0F172A;">{grp_name}</strong> &nbsp;({comp}/{tot} Completed)</span>
                <span style="color: {color}; font-weight: 700;">{pct:.1f}%</span>
            </div>
            <div style="background: #E2E8F0; border-radius: 6px; height: 10px; overflow: hidden; display: flex;">
                <div style="background: #10B981; width: {pct}%; transition: width 0.3s;"></div>
                <div style="background: #EF4444; width: {rerun_pct}%;"></div>
            </div>
            <div style="display: flex; gap: 14px; font-size: 11px; color: #64748B; margin-top: 5px;">
                <span>✅ Completed: <strong style="color: #059669;">{comp}</strong></span>
                <span>⏳ Pending: <strong style="color: #D97706;">{pend}</strong></span>
                <span>⚠️ Needs Rerun: <strong style="color: #DC2626;">{rerun}</strong></span>
                <span>🎯 Queue: <strong style="color: #2563EB;">{q}</strong></span>
            </div>
        </div>
"""
    html += """
    </div>
</div>
"""
    return html

# 2. Audit champions workload (general overview by model)
df_champ_audit = eval_service.audit_champions_workload()
print(f"🏆 Champions Workload: {len(df_champ_audit)} conditions ({df_champ_audit["status"].value_counts().to_dict()})")
display(HTML(render_html_dashboard(df_champ_audit, title="🏆 LLaMEA Champions Evaluation Audit", subtitle="Empirical benchmark evaluation status across LLM-evolved algorithms", group_column="model")))

# 3. Discover unique conditions for classical baselines (general overview by solver)
UNIQUE_CONFIGS = sorted(list({(c["dim"], c["noise_std"], c["problem_id"]) for c in champions_flat.values()}))
df_base_audit = eval_service.audit_baselines_workload()
print(f"⚙️ Baselines Workload: {len(df_base_audit)} conditions ({df_base_audit["status"].value_counts().to_dict()})")
display(HTML(render_html_dashboard(df_base_audit, title="⚙️ Classical Baselines Evaluation Audit", subtitle="Empirical benchmark evaluation status across CMA-ES, DE, and PSO", group_column="solver")))

🏆 Champions Workload: 227 conditions ({'COMPLETED': 227})


⚙️ Baselines Workload: 90 conditions ({'COMPLETED': 90})


## 3. Execute LLaMEA Champions Benchmark (N=20 Independent Runs)
Evaluate evolved algorithms across target BBOB functions with full IOHprofiler `.dat` and `.json` logging.

In [32]:
results_champ_df = eval_service.run_champions()
print(f'✅ LLaMEA Champions evaluation complete: {len(results_champ_df)} conditions processed.')



🚀 STARTING BENCHMARK EVALUATIONS FOR CHAMPIONS
   227 target conditions to evaluate

[1/227] 🏆 Champion: qwen2.5-coder-14b-instruct-q4_k_m.gguf (baseline) | Dim: 2D | Noise: clean (σ=0.0) | Problem: f1 (Sphere (f1))
  📦 [CACHED] 20 runs found (Median Error: -3.3290e+01). Skipping.

[2/227] 🏆 Champion: qwen2.5-coder-14b-instruct-q4_k_m.gguf (baseline) | Dim: 2D | Noise: noisy (σ=0.05) | Problem: f1 (Sphere (f1))
  📦 [CACHED] 20 runs found (Median Error: -3.3289e+01). Skipping.

[3/227] 🏆 Champion: qwen2.5-coder-14b-instruct-q4_k_m.gguf (guided) | Dim: 2D | Noise: clean (σ=0.0) | Problem: f1 (Sphere (f1))
  📦 [CACHED] 20 runs found (Median Error: -3.3290e+01). Skipping.

[4/227] 🏆 Champion: qwen2.5-coder-14b-instruct-q4_k_m.gguf (guided) | Dim: 2D | Noise: noisy (σ=0.05) | Problem: f1 (Sphere (f1))
  📦 [CACHED] 20 runs found (Median Error: -3.3276e+01). Skipping.

[5/227] 🏆 Champion: qwen2.5-coder-14b-instruct-q4_k_m.gguf (thinking) | Dim: 2D | Noise: clean (σ=0.0) | Problem: f1 (Sphere

## 4. Execute Classical Baselines Benchmark (CMA-ES, DE, PSO)
Run classical baseline optimizers across identical problem conditions for rigorous comparative benchmarking.

In [33]:
results_base_df = eval_service.run_baselines()
print(f'✅ Classical Baselines evaluation complete: {len(results_base_df)} conditions processed.')



🚀 STARTING BENCHMARK EVALUATIONS FOR BASELINES
   90 target conditions to evaluate

[1/90] ⚙️ Baseline: CMA-ES | Dim: 2D | Noise: clean (σ=0.0) | Problem: f1 (Sphere (f1))
  📦 [CACHED] 20 runs found (Median Error: -3.3290e+01). Skipping.

[2/90] ⚙️ Baseline: CMA-ES | Dim: 2D | Noise: clean (σ=0.0) | Problem: f8 (Rosenbrock (f8))
  📦 [CACHED] 20 runs found (Median Error: -4.6925e+01). Skipping.

[3/90] ⚙️ Baseline: CMA-ES | Dim: 2D | Noise: clean (σ=0.0) | Problem: f11 (Discus (f11))
  📦 [CACHED] 20 runs found (Median Error: -1.7625e+01). Skipping.

[4/90] ⚙️ Baseline: CMA-ES | Dim: 2D | Noise: clean (σ=0.0) | Problem: f15 (Rastrigin Multi-Modal (f15))
  📦 [CACHED] 20 runs found (Median Error: 7.7223e+01). Skipping.

[5/90] ⚙️ Baseline: CMA-ES | Dim: 2D | Noise: clean (σ=0.0) | Problem: f21 (Gallagher 101 Peaks (f21))
  📦 [CACHED] 20 runs found (Median Error: -2.8794e+00). Skipping.

[6/90] ⚙️ Baseline: CMA-ES | Dim: 2D | Noise: noisy (σ=0.05) | Problem: f1 (Sphere (f1))
  📦 [CACHED] 2